In [ ]:
# Cell 1: Environment Setup

include("helpers/initialization_helpers.jl")
include("helpers/initialization_dictionaries.jl")
using .InitializationHelpers
using .InitializationDictionaries
using OMJulia
using Plots, DataFrames, CSV

# --- Configuration ---

# 1. Directory containing the original dynamic package
MODEL_DIR = abspath("MyNordic")
MODELS_PKG_PATH = joinpath(MODEL_DIR, "package.mo")

# 2. Directory containing the auxiliary package generated by BuildAux_package.ipynb
AUX_DIR = abspath("MyNordic_auxiliary")
AUX_PACKAGE_FILE = joinpath(AUX_DIR, "package.mo")

# 3. Root dynamic model to initialize
MODEL = "MyNordic.TestCase"

# 4. Derived package/model names
SOURCE_PACKAGE = split(MODEL, ".")[1]
ROOT_MODEL_NAME = split(MODEL, ".")[end]

AUX_PACKAGE = SOURCE_PACKAGE * "_auxiliary"
AUX_ROOT_MODEL = AUX_PACKAGE * "." * ROOT_MODEL_NAME * "_auxiliary"

INITIALIZED_PACKAGE = SOURCE_PACKAGE * "_initialized"
INITIALIZED_DIR = joinpath(dirname(MODEL_DIR), INITIALIZED_PACKAGE)
INITIALIZED_ROOT_MODEL = INITIALIZED_PACKAGE * "." * ROOT_MODEL_NAME * "_initialized"
INITIALIZED_PACKAGE_FILE = joinpath(INITIALIZED_DIR, "package.mo")
INITIALIZED_ORDER_FILE = joinpath(INITIALIZED_DIR, "package.order")

# 5. Library paths
DYNAWO_PKG_PATH = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

# 6. Placeholder plot variable for the final initialized simulation
PLOT_VARIABLE = "g20.terminal.V.re"


In [ ]:
# Cell 2: Package helper functions

function inherited_classes(omc, model::String)
    raw = strip(String(om_send(omc, "getInheritedClasses($model)", parsed = false)))
    raw = replace(raw, "{" => "", "}" => "", ";" => "", "\"" => "")
    isempty(strip(raw)) && return String[]

    parents = String[]
    for parent in split(raw, ",")
        parent = strip(parent)
        isempty(parent) && continue
        startswith(parent, "Modelica.Icons") && continue
        startswith(parent, "Dynawo.Icons") && continue
        push!(parents, parent)
    end

    return parents
end

function get_inheritance_chain(omc, root::String)
    chain = String[]
    seen = Set{String}()

    function visit(model::String)
        model in seen && return
        push!(seen, model)

        for parent in inherited_classes(omc, model)
            visit(parent)
        end

        push!(chain, model)
    end

    visit(root)
    return chain
end

function rewrite_initialized_extends(txt::String, initialized_name_map::Dict{String, String})
    rewritten = txt
    for (original_parent, initialized_parent) in initialized_name_map
        rewritten = replace(rewritten, "extends " * original_parent * ";" => "extends " * initialized_parent * ";")
        rewritten = replace(rewritten, "extends " * original_parent * "(" => "extends " * initialized_parent * "(")
    end
    return rewritten
end


In [ ]:
# Cell 3: OpenModelica Setup + Package Loading

# 1. Load the original dynamic package
SourceOMC = OMJulia.OMCSession()
om_send(SourceOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
om_send(SourceOMC, "loadModel(Complex)")
om_send(SourceOMC, "loadModel(ModelicaServices)")
om_send(SourceOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
om_send(SourceOMC, "loadFile(\"$MODELS_PKG_PATH\")")
om_send(SourceOMC, "clearMessages()")
println("Checking the dynamic root model...")
chk_source = om_send(SourceOMC, "checkModel($MODEL)", parsed=false)
println(chk_source)

# 2. Load the auxiliary package
AuxOMC = OMJulia.OMCSession()
om_send(AuxOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
om_send(AuxOMC, "loadModel(Complex)")
om_send(AuxOMC, "loadModel(ModelicaServices)")
om_send(AuxOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
om_send(AuxOMC, "loadFile(\"$AUX_PACKAGE_FILE\")")
om_send(AuxOMC, "clearMessages()")
println("Checking the auxiliary root model...")
chk_aux = om_send(AuxOMC, "checkModel($AUX_ROOT_MODEL)", parsed=false)
println(chk_aux)


In [ ]:
# Cell 4: Inheritance Chain + Package Name Mapping

chain = get_inheritance_chain(SourceOMC, MODEL)
println("Inheritance chain:")
println(chain)

AUX_NAME_MAP = Dict(model => AUX_PACKAGE * "." * split(model, ".")[end] * "_auxiliary" for model in chain)
INITIALIZED_NAME_MAP = Dict(model => INITIALIZED_PACKAGE * "." * split(model, ".")[end] * "_initialized" for model in chain)

println("\nInitialized class map:")
for model in chain
    println("  ", model, " -> ", INITIALIZED_NAME_MAP[model])
end


In [ ]:
# Cell 5: Simulate Auxiliary Package + Extract Initialization Values

# Build and simulate the auxiliary root model.
ModelicaSystem(AuxOMC, AUX_PACKAGE_FILE, AUX_ROOT_MODEL, [MODELICA_PKG_PATH, DYNAWO_PKG_PATH])
simulate(AuxOMC, resultfile = AUX_PACKAGE * "_res.mat")

# Extract initialization values per original class in the inheritance chain.
components_by_model = Dict{String, Dict{String, Dict{String, Any}}}()
initializable_by_model = Dict{String, Dict{String, Dict{String, Any}}}()
init_values_by_model = Dict{String, Dict{String, Dict{String, Float64}}}()

for model in chain
    components = get_all_components(SourceOMC, model)
    initializable_components = get_initializable_components(components, INIT_PARAMS)
    init_values_by_component = extract_all_initialization_values(AuxOMC, initializable_components, INIT_PARAMS)

    components_by_model[model] = components
    initializable_by_model[model] = initializable_components
    init_values_by_model[model] = init_values_by_component

    println(model, ": ", length(initializable_components), " initializable component(s)")
end


In [ ]:
# Cell 6: Build and save the initialized package
om_send(SourceOMC, "deleteClass($INITIALIZED_PACKAGE)")
om_send(SourceOMC, "clearMessages()")
om_send(SourceOMC, "loadString(\"within ; package $INITIALIZED_PACKAGE end $INITIALIZED_PACKAGE;\")")

for model in chain
    initialized_model = INITIALIZED_NAME_MAP[model]
    initialized_name = split(initialized_model, ".")[end]

    println("Copying and initializing ", model, " -> ", initialized_model)
    om_send(SourceOMC, "copyClass($model, \"$initialized_name\", $INITIALIZED_PACKAGE)")

    apply_initialization_modifiers!(
        SourceOMC,
        initialized_model,
        initializable_by_model[model],
        INIT_PARAMS,
        init_values_by_model[model],
    )
end

mkpath(INITIALIZED_DIR)

open(INITIALIZED_PACKAGE_FILE, "w") do io
    print(io, "within ;\npackage $INITIALIZED_PACKAGE\nend $INITIALIZED_PACKAGE;\n")
end

open(INITIALIZED_ORDER_FILE, "w") do io
    for model in chain
        println(io, split(INITIALIZED_NAME_MAP[model], ".")[end])
    end
end

for model in chain
    initialized_model = INITIALIZED_NAME_MAP[model]
    initialized_name = split(initialized_model, ".")[end]
    initialized_file = joinpath(INITIALIZED_DIR, initialized_name * ".mo")

    txt = String(om_send(SourceOMC, "listFile($initialized_model)"))
    txt = rewrite_initialized_extends(txt, INITIALIZED_NAME_MAP)

    open(initialized_file, "w") do io
        print(io, txt)
        endswith(txt, "\n") || print(io, "\n")
    end
end

InitializedOMC = OMJulia.OMCSession()
om_send(InitializedOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
om_send(InitializedOMC, "loadModel(Complex)")
om_send(InitializedOMC, "loadModel(ModelicaServices)")
om_send(InitializedOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
om_send(InitializedOMC, "loadFile(\"$INITIALIZED_PACKAGE_FILE\")")
om_send(InitializedOMC, "clearMessages()")
println("Checking the initialized root model...")
chk_initialized = om_send(InitializedOMC, "checkModel($INITIALIZED_ROOT_MODEL)", parsed=false)
println(chk_initialized)
